# GTR + RADAR v6 on Colab (Python 3.12, single A100, fp32 + TF32)

1. Runtime → Change runtime type → **A100 GPU**.
2. Push the repo to GitHub (or upload it to Drive).
3. Put the original dataset files in a Drive folder (`ETTh1.csv`, `ETTh2.csv`, `ETTm1.csv`, `ETTm2.csv`, `weather.csv`, `electricity.csv`, `traffic.csv`). Nothing is generated synthetically; a missing file stops the pipeline.

v6 = v5 + cross-channel expert E4 + segment-level gate + forecast-space mixing (`--arch v6`, the default; `ARCH=v5` / `ARCH=v4` run the earlier architectures).
Checkpoints are reused when they exist (`SKIP_EXISTING=1`): `base.pth` and `aug_v4.pth` are shared by all architectures; RADAR checkpoints are `pre_v6.pth` / `plugin_v6.pth` (v5: `*_v5.pth`). Results go to `results_radar_v6/`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python --version
import torch, numpy, pandas, sklearn
print('torch', torch.__version__, '| numpy', numpy.__version__, '| pandas', pandas.__version__, '| sklearn', sklearn.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/knull-cc/GTR_RADAR.git'      # or copy the repo from Drive instead
DATA_SRC = '/content/drive/MyDrive/datasets'                 # folder holding the dataset files
SAVE_DIR = '/content/drive/MyDrive/GTR_RADAR_runs'           # results / figures / logs / checkpoints

In [ ]:
%cd /content
!test -d GTR_RADAR || git clone $REPO_URL GTR_RADAR
%cd /content/GTR_RADAR
!git log --oneline -1
!mkdir -p dataset && cp -rn "$DATA_SRC"/* dataset/ 2>/dev/null; ls -R dataset | head -40
import os
os.environ['CKPT_DIR'] = SAVE_DIR + '/checkpoints_radar/'   # survives disconnects; base.pth is reused from here

## 0 · Unit tests (B=4, L=96, C=7, H=96; random GTR in eval mode)

In [ ]:
!pip -q install pytest
!python -m pytest -q tests/test_radar.py

## 1 · Step 1 — RevIN 2×2 on B0 (validation split, H=96)
Clean / Raw / Stats-only / Direct-only forecast MSE per type × position. Decides the gate / dependency branch:
Internal Raw ≈ Clean → keep the gate; damage mostly *Stats-only* → add the repair-size term; mostly *Direct-only* → per-sample dependency.

In [ ]:
!bash scripts/RADAR/run_step.sh revin ETTh1
from IPython.display import Markdown, display
import glob
for f in sorted(glob.glob('results_radar_v6/diag/*__revin.md')):
    display(Markdown(open(f).read()))

## 2 · Step 2 — offline experts (oracle detection, no training)
Repair MAE of no-repair / local median / E2 / E3 / E4 (cross-channel) / best-of on the corrupted points, and the B0 forecast MSE after replacement.
ETTh1/ECL/Traffic have E3 (T = 24); ETTm (T = 96) and Weather (T = 144) do not (T ≥ L = 96). E4 needs reliable partner channels: it is unavailable on the `c100` conditions (all channels corrupted at the same time) and available on `c50`.

In [ ]:
!bash scripts/RADAR/run_step.sh experts ETTh1
for f in sorted(glob.glob('results_radar_v6/diag/*__experts.md')):
    display(Markdown(open(f).read()))

## 3 · Step 3 — detector features v4 vs v3 (S2 only, AUROC per type × position)

In [ ]:
!bash scripts/RADAR/run_step.sh det ETTh1
!DET_FEATS=v3 bash scripts/RADAR/run_step.sh det ETTh1
for f in sorted(glob.glob('results_radar_v6/diag/*__det_det*.md')):
    display(Markdown(open(f).read()))

## 4 · Step 4 — main experiment
Quick check on ETTh1 first (H=96), then the 7 datasets × H ∈ {96, 192, 336, 720}, seed 2024.
`ORACLE=1` also evaluates the plug-in with a perfect gate (**Ours-oracle**: how much a better gate could still gain); `ARGMAX=1` hard routing; `RADAR_ARGS="--test_grid main"` halves the evaluation time.

In [ ]:
!ORACLE=1 REPRO=0 bash scripts/RADAR/run_quick.sh ETTh1   # REPRO=1 re-runs the original run.py check
display(Markdown(open('results_radar_v6/summary.md').read()))

Add the v5 plug-in (already trained, `plugin_v5.pth` in `CKPT_DIR`) to the same results as **Ours-plugin-v5**; the Δ table then also shows v6 vs v5:

In [ ]:
!ARCH=v5 METHODS=Ours-plugin MODE=eval QUICK=1 REPRO=0 bash scripts/RADAR/etth1.sh
display(Markdown(open('results_radar_v6/summary.md').read()))

In [ ]:
!bash scripts/RADAR/run_all.sh etth1 etth2 ettm1 ettm2
!mkdir -p "$SAVE_DIR" && cp -r results_radar_v6 figures_radar_v6 logs_radar_v6 "$SAVE_DIR"/

In [ ]:
!bash scripts/RADAR/run_all.sh weather electricity traffic
!mkdir -p "$SAVE_DIR" && cp -r results_radar_v6 figures_radar_v6 logs_radar_v6 "$SAVE_DIR"/
display(Markdown(open('results_radar_v6/summary.md').read()))
display(Markdown(open('results_radar_v6/summary_router.md').read()))